# The Dream

> What if we never needed to worry about kv-caches, flash-attention, pipeline parallelisim, and more?

I hate that so much time, effort, and money had been spent on things that can be fully automated.

In the world of LLMs, we should be able to fully automate techniques like kv-caching, flash attention, pipeline parallelsim, and more. 

How?

We just need a smart compiler.

Let me explain.

Consider a tiny dataset `X` with a batch size of 1 and a tiny transformer based LLM `M` that has one layer and one attention head. We train this model on a single GPU. In for loops, this is

In [ ]:
attn(X, M)

In [ ]:
for d in dim:
    for token in seqs:
        X[d, token]
        M[d, token]

Consider a standard transformer based LLM. The fundamental unit here is the attention head. Every attention head operates independently. Independently over what? Independently over gpus, dataset batch, model layers, and other attention heads. In for-loops:

In [ ]:
for gpu in gpus: # G
    for batch in dataset: # X
        for layer in model_layers: # L
            for attn_head in attn_heads: # H
                # forward pass operates on a tensor of dimension (B, H, T, Dh)
                attn(gpu, batch, layer, attn_head)

where `G` is the number of GPUs, `X` is the size of the dataset, `L`, is the number of layers in the model, `H` is the number of attention heads, `B` is the batch size, `T` is the sequence length in tokens, and `Dh` is the head dimension. Recall that the hidden dimension is `D=H*Dh`.

Normally, attention is not expressed as for loops, but I like it better this way.

Recall that attention operates on a tensor with dimensions `(B,T,D)` where `B` is the batch size, `T` is the sequence length in tokens, `D` is the hidden dimension. We can

In [ ]:
x = dataset[gpu, batch, sequence, token]
w = model[gpu, layer, attn_head, one_dim]

attn(x, w)

Let's zoom in to the attention bottleneck: the query key multiplication. Every query and key is multiplied independently too. In for-loops

In [ ]:
for gpu in gpus:
    for batch in dataset:
        for layer in model_layers:
            for attn_head in attn_heads:
                # some attention stuff
                for query in queries:
                    for key in keys:
                        query.dot(key) # attention bottleneck
                # some attention stuff

We've made explicit 6 different independent operations:
* gpus
* dataset
* model_layers
* attn_heads
* queries
* keys

The kv cache works by changing the for-loop order and storing some things.

In [ ]:
for gpu in gpus:
    for batch in dataset:
        for layer in model_layers:
            for attn_head in attn_heads:
                # some attention stuff
                for query in queries:
                    for key in keys:
                        query.dot(key) # attention bottleneck
                        for token in seq:
                            for d in dim:
                                f(query[d], key[d], mask[d, seq])
                # some attention stuff

What about hardware for loops?

In [ ]:
for gpu in gpus:
    for batch in dataset:
        for layer in model_layers:
            for attn_head in attn_heads:
                # some attention stuff
                for query in queries:
                    for key in keys:
                        query.dot(key) # attention bottleneck
                # some attention stuff